In [ ]:
import torch                    # PyTorch 主库：张量运算、自动求导、神经网络
from torch import nn            # nn = neural networks，提供各类网络层与损失函数
from d2l import torch as d2l    # d2l 工具库：封装了数据加载、画图、训练循环等常用函数




In [ ]:
# -----------------------------------------------
# 1) 超参数（hyperparameters）与数据集加载
# -----------------------------------------------
# batch_size（批量大小）：一次“喂给”模型多少张图片进行一次前向/反向与参数更新。
# 常见取值：64、128、256… 越大单次更稳定，但显存/内存占用更高。
batch_size = 256
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size)
# d2l.load_data_fashion_mnist 会自动：
#  - 下载/读取 Fashion-MNIST（28×28 灰度图，十类：T恤、鞋、包等）
#  - 把图片转成 PyTorch Tensor，并做基础预处理（如 ToTensor，像素缩放到[0,1]）
#  - 返回两个 DataLoader：
#       train_iter —— 训练集迭代器（会被反复遍历来更新参数）
#       test_iter  —— 测试集迭代器（只用来评估效果，不更新参数）

In [ ]:
# -----------------------------------------------
# 2) 定义网络结构（model / net）
# -----------------------------------------------
# nn.Sequential 按顺序把多层“拼接”起来：
#   - nn.Flatten(): 把 28×28 的图片摊平成长度 784 的一维向量
#   - nn.Linear(784, 10): 全连接层（又叫仿射层/稠密层），输出 10 个“类别打分”(logits)
# 注意：这里没有显式写 Softmax，因为 CrossEntropyLoss 内部会完成 log_softmax。
net = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 10)
)

# 权重初始化：良好的初始化能帮助模型更快更稳定地收敛。
# 下面定义一个函数，等会儿用 net.apply 把它应用到每一层。


In [ ]:
net=nn.Sequential(nn.Flatten(),nn.Linear(784,10))
def init_weights(m):
    """若当前子模块 m 是线性层 Linear，则用 N(0, 0.01) 初始化其权重。"""
    if isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, std=0.01)  # 均值 0，标准差 0.01 的正态分布
        # 对于 bias（偏置），PyTorch 一般会用 0 初始化；保持默认即可。

net.apply(init_weights);


In [ ]:
# -----------------------------------------------
# 3) 定义损失函数（loss）与优化器（optimizer）
# -----------------------------------------------
# CrossEntropyLoss 适用于“单标签多分类”（one-hot 中只有一个 1 的情形）。
# 它内部等价于：log_softmax + NLLLoss（负对数似然损失），所以模型输出只需给 logits。
# reduction 参数控制“如何汇总这个 batch 里每个样本的损失”：
#   - 'none'：不汇总，返回形状为 (batch_size,) 的逐样本损失（d2l 的训练函数会再做平均）
#   - 'mean'：对 batch 内样本求平均（得到标量）
#   - 'sum' ：对 batch 内样本求和（得到标量）
loss = nn.CrossEntropyLoss(reduction='none')
trainer = torch.optim.SGD(net.parameters(), lr=0.1)

# 优化器：SGD（随机梯度下降）。
#  - net.parameters(): 需要被更新的全部参数（权重、偏置）
#  - lr=0.1：学习率（每次更新“迈多大步”）。过大可能发散，过小可能收敛很慢。


In [ ]:
# -----------------------------------------------
# 4) 训练与评估（training & evaluation）
# -----------------------------------------------
# epoch：把训练集“完整看一遍”称为 1 个 epoch。这里我们训练 10 个 epoch。
num_epochs = 10

In [ ]:
# d2l 中的 train_ch3 会：
#   - 遍历 train_iter 做前向、计算损失、反向传播与参数更新
#   - 定期在 test_iter 上评估准确率
#   - 绘制训练曲线（若在交互环境中）
# 如果你遇到 NameError: name 'train_ch3' is not defined，
# 请将下一行替换为： d2l.train_ch3(net, train_iter, test_iter, loss, num_epochs, trainer)

In [ ]:
for epoch in range(num_epochs):
    net.train()  # 开启训练模式（对 Dropout、BN 等有效）
    total_loss, total_correct, total_num = 0.0, 0, 0

    # 遍历训练数据集的每一个 batch
    for X, y in train_iter:
        y_hat = net(X)                     # 1) 前向传播：计算预测结果
        l = loss(y_hat, y).mean()          # 2) 计算平均损失

        trainer.zero_grad()                # 3) 清除旧的梯度
        l.backward()                       # 4) 反向传播：计算梯度
        trainer.step()                     # 5) 更新参数

        total_loss += l.item() * X.shape[0] # 累积损失（乘上 batch_size）
        preds = y_hat.argmax(dim=1)         # 预测类别（取最大概率对应的类别）
        total_correct += (preds == y).sum().item()
        total_num += X.shape[0]

    train_loss = total_loss / total_num
    train_acc = total_correct / total_num

    # 在测试集上评估准确率
    net.eval()  # 切换到评估模式
    test_correct, test_num = 0, 0
    with torch.no_grad():
        for X, y in test_iter:
            preds = net(X).argmax(dim=1)
            test_correct += (preds == y).sum().item()
            test_num += X.shape[0]
    test_acc = test_correct / test_num

    print(f"Epoch {epoch+1}/{num_epochs} | Train loss: {train_loss:.4f} | Train acc: {train_acc:.4f} | Test acc: {test_acc:.4f}")